# 14. Supervised Learning: Logistic Regression

## Algorithm Category
**Type**: Supervised Learning - Classification  
**Complexity**: Low  
**Use Case**: Binary and multiclass classification

## Learning Objectives

By the end of this notebook, you will be able to:
- Understand the mathematical foundation of logistic regression
- Implement logistic regression using scikit-learn
- Evaluate classification performance using multiple metrics
- Interpret model coefficients and probabilities
- Apply logistic regression to real-world classification problems

## Historical Context

Logistic regression was developed in the 1950s and 1960s, building on earlier work in statistics. The logistic function (sigmoid) was first used by Pierre François Verhulst in the 19th century to model population growth. The application to binary classification was formalized by statisticians including David Cox.

**Key Papers/References:**
- Cox, D.R. (1958). "The regression analysis of binary sequences"
- Verhulst, P.F. (1838). "Notice sur la loi que la population suit dans son accroissement"

## When to Use Logistic Regression

Logistic regression is appropriate when:
- The target variable is categorical (binary or multiclass)
- You need probability estimates, not just class predictions
- Interpretability is important (coefficients show feature impact)
- The relationship between features and log-odds is approximately linear
- Dataset size is moderate (works well with thousands to millions of samples)

## Theory & Mechanics

### Mathematical Foundation

Logistic regression models the probability that a sample belongs to a particular class using the logistic (sigmoid) function:

**Binary Classification:**
$$P(y=1|X) = \frac{1}{1 + e^{-(\beta_0 + \beta_1 x_1 + ... + \beta_n x_n)}} = \sigma(z)$$

Where $\sigma(z)$ is the sigmoid function and $z = \beta_0 + \beta_1 x_1 + ... + \beta_n x_n$

**Log-Odds (Logit):**
$$\log\left(\frac{P(y=1|X)}{1-P(y=1|X)}\right) = \beta_0 + \beta_1 x_1 + ... + \beta_n x_n$$

### How It Works

1. **Objective**: Find coefficients that maximize the likelihood of observing the training data
2. **Cost Function**: Cross-entropy (log loss)
   $$L = -\frac{1}{n}\sum_{i=1}^{n}[y_i \log(\hat{p}_i) + (1-y_i)\log(1-\hat{p}_i)]$$
3. **Optimization**: Typically uses gradient descent or Newton's method (no closed-form solution)
4. **Prediction**: Outputs probabilities, which are then thresholded (usually 0.5) to get class predictions

### Key Assumptions

1. **Linearity**: Log-odds are linear in features
2. **Independence**: Observations are independent
3. **No multicollinearity**: Features should not be highly correlated
4. **Large sample size**: Works better with more data

### Limitations

- Assumes linear decision boundary (can be extended with polynomial features)
- Sensitive to outliers
- Requires feature scaling for convergence
- May struggle with non-linear relationships


## Implementation

Let's implement logistic regression step by step using scikit-learn and our helper functions.

**Implementation Steps:**
1. **Import Libraries**: Load necessary tools and datasets
2. **Load Data**: Get classification dataset (binary or multiclass)
3. **Preprocess**: Scale features, split data
4. **Train Model**: Fit logistic regression to training data
5. **Make Predictions**: Get class predictions and probabilities
6. **Evaluate**: Calculate classification metrics (accuracy, precision, recall, F1)
7. **Validate**: Check model assumptions and performance
8. **Interpret**: Understand coefficients and feature importance


In [ ]:
# ============================================
# IMPORTING LIBRARIES: Setting Up Our Tools
# ============================================

# Core data science libraries
import numpy as np  # NumPy: Numerical computing (arrays, math operations)
import pandas as pd  # Pandas: Data manipulation (DataFrames, data analysis)
import matplotlib.pyplot as plt  # Matplotlib: Plotting and visualization

# Scikit-learn: Machine learning library
from sklearn.datasets import load_breast_cancer, load_iris  # Classification datasets
from sklearn.linear_model import LogisticRegression  # Logistic regression algorithm
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV  # Model selection tools
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix  # Classification metrics

# ============================================
# IMPORTING OUR HELPER FUNCTIONS
# ============================================

# Our custom utility functions (organized in src/ directory)
from src.models.supervised import split_data, evaluate_classifier, cross_validate_model  # Supervised learning utilities
from src.models.classification import (
    calculate_classification_metrics,  # Calculate precision, recall, F1, etc.
    plot_confusion_matrix,  # Visualize confusion matrix
    plot_roc_curve, plot_precision_recall_curve, analyze_confusion_matrix
)
from src.processing.preprocessing import scale_features
from src.utils.benchmarking import benchmark_model_training
from src.utils.traceability import extract_feature_importance_trace, save_traceability_data
from src.utils.validation import validate_model_output, check_cross_validation_stability

print("Libraries imported successfully!")


In [ ]:
# ============================================
# LOADING THE DATASET: Breast Cancer Classification
# ============================================

# load_breast_cancer() loads the Breast Cancer Wisconsin dataset from scikit-learn
# This is a binary classification problem: predict if tumor is malignant (1) or benign (0)
# No download needed - it's built into scikit-learn
cancer = load_breast_cancer()  # Returns a Bunch object with data, target, feature_names

# X = Features (inputs): Medical measurements of breast tumors
# cancer.data contains feature values (569 samples × 30 features)
# We convert to DataFrame for easier manipulation
# columns=cancer.feature_names gives meaningful column names (mean radius, mean texture, etc.)
X = pd.DataFrame(cancer.data, columns=cancer.feature_names)
# Features include: mean radius, mean texture, mean perimeter, mean area, etc. (30 total)

# y = Target (output): Tumor type (what we want to predict)
# cancer.target contains class labels (0 = benign, 1 = malignant)
# We convert to Series and give it a descriptive name
y = pd.Series(cancer.target, name='Target')
# 0 = benign (non-cancerous), 1 = malignant (cancerous)

# ============================================
# EXPLORING THE DATASET: Understanding Our Data
# ============================================

# .shape returns (rows, columns) - dimensions of the dataset
print(f"Dataset Shape: {X.shape}")  # Output: (569, 30) - 569 patients, 30 features

# Display class names (what the target values represent)
print(f"Classes: {cancer.target_names}")  # Output: ['malignant' 'benign']

# .value_counts() counts how many samples belong to each class
# .to_dict() converts to dictionary for easier display
print(f"Class distribution: {y.value_counts().to_dict()}")  # Shows: {0: X, 1: Y} (class counts)

# Display first few rows to see what the data looks like
print(f"\nFirst few rows:")
print(X.head())  # Shows first 5 rows with all feature values


In [ ]:
# ============================================
# FEATURE SCALING: Normalizing Features
# ============================================

# Why scale features for logistic regression?
# - Logistic regression uses gradient descent optimization
# - Features on different scales cause slow convergence or non-convergence
# - Scaling ensures all features contribute equally to the model
# - Improves numerical stability during optimization

# scale_features() normalizes features to have mean=0 and std=1
# fit=True means "learn scaling from this data" (use for training data)
# Returns: scaled features and the scaler (save scaler to scale test data later)
X_scaled, scaler = scale_features(X, fit=True)
# X_scaled: Features normalized (mean=0, std=1 for each column)
# scaler: The scaling object (needed to scale test data with same transformation)

# ============================================
# TRAIN/TEST SPLIT: Separating Data
# ============================================

# CRITICAL: Split AFTER scaling to prevent data leakage
# If we split first, test data statistics could influence scaling

# split_data() randomly splits data into training (80%) and test (20%) sets
# test_size=0.2 means 20% for testing, 80% for training
# random_state=42 ensures same split every time (reproducibility)
X_train, X_test, y_train, y_test = split_data(X_scaled, y, test_size=0.2, random_state=42)
# X_train: 80% of samples for training (scaled features)
# X_test: 20% of samples for testing (will be scaled using same scaler later)
# y_train: Target values for training samples
# y_test: Target values for test samples (ground truth)

# Display split information
print(f"Training set: {X_train.shape[0]} samples")  # ~455 samples (80% of 569)
print(f"Test set: {X_test.shape[0]} samples")  # ~114 samples (20% of 569)


In [ ]:
# ============================================
# MODEL CREATION: Logistic Regression
# ============================================

# Create a LogisticRegression model object
# This model will learn a logistic relationship: P(y=1) = σ(β₀ + β₁x₁ + ... + βₙxₙ)
# max_iter=1000: Maximum iterations for optimization (may need more for complex problems)
# random_state=42: Ensures reproducible results
model = LogisticRegression(max_iter=1000, random_state=42)

# ============================================
# MODEL TRAINING: Learning from Data
# ============================================

# .fit() trains the model on training data
# The model learns the coefficients (β values) that maximize likelihood
# Internally uses optimization (gradient descent or Newton's method)
model.fit(X_train, y_train)  # Train the model

print("Model trained successfully!")  # Confirm training completed

# ============================================
# MODEL INTERPRETATION: Understanding Coefficients
# ============================================

# model.intercept_ is β₀ (the log-odds when all features are 0)
# For binary classification, intercept_ is an array with one value
# [0] gets the first (and only) element
print(f"Intercept: {model.intercept_[0]:.3f}")  # Display intercept value

# model.coef_ contains β₁, β₂, ..., βₙ (coefficients for each feature)
# For binary classification, coef_ is a 2D array with shape (1, n_features)
# [0] gets the first row (coefficients for class 1)
# Each coefficient tells us: "How much does log-odds change when this feature increases by 1?"
# Positive coefficient: feature increases → higher probability of class 1
# Negative coefficient: feature increases → lower probability of class 1

print(f"\nTop 5 Coefficients (by absolute value):")
# Create DataFrame to display coefficients nicely
coef_df = pd.DataFrame({
    'Feature': X.columns,  # Feature names
    'Coefficient': model.coef_[0]  # Coefficient values (β values for class 1)
})

# Calculate absolute value of coefficients (importance magnitude)
coef_df['Abs_Coefficient'] = coef_df['Coefficient'].abs()

# Sort by absolute value (largest impact first)
coef_df = coef_df.sort_values('Abs_Coefficient', ascending=False)
print(coef_df.head())  # Display top 5 most important features


In [ ]:
# ============================================
# MAKING PREDICTIONS: Using the Trained Model
# ============================================

# .predict() uses the trained model to make class predictions
# Returns predicted class (0 or 1 for binary classification)
# Internally: calculates probability, then thresholds at 0.5
y_pred = model.predict(X_test)  # Predictions: array of 0s and 1s

# .predict_proba() returns probability estimates for each class
# Returns 2D array: [probability_class_0, probability_class_1] for each sample
# [:, 1] gets the second column (probability of class 1 = positive class)
y_pred_proba = model.predict_proba(X_test)[:, 1]  # Probabilities: array of values between 0 and 1

# ============================================
# MODEL EVALUATION: Measuring Performance
# ============================================

# evaluate_classifier() calculates basic classification metrics
results = evaluate_classifier(model, X_test, y_test)

# calculate_classification_metrics() computes comprehensive classification metrics
# Includes: accuracy, precision, recall, F1, ROC-AUC, etc.
metrics = calculate_classification_metrics(y_test.values, y_pred, y_pred_proba)

# ============================================
# DISPLAYING PERFORMANCE METRICS
# ============================================

print("Classification Metrics:")

# Accuracy: (correct predictions) / (total predictions)
# Range: 0 to 1. Higher is better
print(f"  Accuracy: {metrics['accuracy']:.3f}")  # Overall correctness

# Precision: (true positives) / (true positives + false positives)
# "Of all predicted positives, how many were actually positive?"
# Range: 0 to 1. Higher is better
print(f"  Precision: {metrics['precision']:.3f}")  # Precision of positive predictions

# Recall (Sensitivity): (true positives) / (true positives + false negatives)
# "Of all actual positives, how many did we catch?"
# Range: 0 to 1. Higher is better
print(f"  Recall: {metrics['recall']:.3f}")  # Ability to find all positives

# F1 Score: Harmonic mean of precision and recall
# F1 = 2 × (precision × recall) / (precision + recall)
# Range: 0 to 1. Higher is better. Balances precision and recall
print(f"  F1 Score: {metrics['f1_score']:.3f}")  # Balanced metric

# ROC-AUC: Area under the ROC curve
# Measures ability to distinguish between classes
# Range: 0 to 1. 1.0 = perfect, 0.5 = random, <0.5 = worse than random
if 'roc_auc' in metrics:
    print(f"  ROC-AUC: {metrics['roc_auc']:.3f}")  # Classification performance


## Validation & Testing

Let's validate our model using multiple approaches: in-notebook assertions, cross-validation, and performance metrics.

**Why Validate?**
- **Catch Errors**: Ensure model is working correctly
- **Assess Performance**: Verify model meets quality standards
- **Check Stability**: Ensure model performance is consistent
- **Build Confidence**: Multiple validation methods increase trust in results

**Validation Methods:**
1. **Output Validation**: Check predictions are reasonable
2. **Cross-Validation**: Test on multiple train/test splits
3. **Confusion Matrix Analysis**: Understand error patterns
4. **Performance Metrics**: Measure classification quality


In [ ]:
# ============================================
# VALIDATION 1: Model Output Validity
# ============================================

# validate_model_output() checks if predictions make sense
# It verifies:
# - Predictions and true values have same shape
# - Predictions are valid class labels (0 or 1 for binary)
# - No NaN or invalid values in predictions
# - Predictions match expected number of classes

# task_type='classification' tells validator this is a classification problem
validation_result = validate_model_output(y_pred, y_test.values, task_type='classification')
# Returns dictionary with validation results

print("Model Output Validation:")
print(f"  Valid: {validation_result['valid']}")  # True if predictions are valid

# If validation passed, display additional information
if 'accuracy' in validation_result:
    print(f"  Accuracy: {validation_result['accuracy']:.3f}")  # Validation accuracy
    print(f"  Number of classes: {validation_result['n_classes']}")  # Number of classes detected

# ============================================
# ASSERTIONS: Automated Checks
# ============================================

# assert statements automatically check conditions
# If condition is False, program stops with error message

# Check 1: Predictions must be valid
assert validation_result['valid'], "Model predictions are invalid!"
# If predictions are invalid, stop execution with error message

# Check 2: Accuracy must be better than random guessing
# For binary classification, random = 0.5 (50% chance)
assert metrics['accuracy'] > 0.5, "Accuracy should be better than random!"
# If accuracy ≤ 0.5, model is no better than flipping a coin

# Check 3: Accuracy cannot exceed 1.0 (100%)
assert metrics['accuracy'] <= 1.0, "Accuracy cannot exceed 1.0"
# Accuracy > 1.0 would indicate a bug

print("\n✓ Basic validation checks passed")  # All assertions passed!


In [ ]:
# ============================================
# VALIDATION 2: Cross-Validation
# ============================================

# Cross-validation splits data into k folds (groups)
# Trains on k-1 folds, tests on 1 fold
# Repeats k times (each fold used as test set once)
# More reliable than single train/test split

# cross_val_score() performs k-fold cross-validation
# cv=5 means 5 folds (5 train/test splits)
# scoring='accuracy' means use accuracy as the evaluation metric
cv_scores = cross_val_score(model, X_scaled, y, cv=5, scoring='accuracy')
# Returns array of 5 accuracy scores (one per fold)

# Calculate statistics across folds
cv_mean = cv_scores.mean()  # Average accuracy across all folds
cv_std = cv_scores.std()  # Standard deviation (measure of variability)

print("Cross-Validation Results (5-fold):")
print(f"  Mean Accuracy: {cv_mean:.3f} (+/- {cv_std:.3f})")  # Average ± variability
print(f"  Individual fold accuracies: {cv_scores}")  # Accuracy for each of the 5 folds

# ============================================
# STABILITY CHECK: Is Performance Consistent?
# ============================================

# Check if cross-validation results are stable (low variation)
# Unstable results suggest model is sensitive to data split
# Stable results suggest model is robust

# check_cross_validation_stability() calculates coefficient of variation
# threshold=0.1 means "variation should be less than 10% of mean"
stability = check_cross_validation_stability(cv_scores, threshold=0.1)
# Returns dictionary with stability analysis

print(f"\nCV Stability Check:")
print(f"  Coefficient of Variation: {stability['cv_coefficient']:.3f}")  # Relative variability
print(f"  Is Stable: {stability['is_stable']}")  # True if variation < threshold

# ============================================
# ASSERTIONS: Cross-Validation Checks
# ============================================

# Check 1: CV accuracy must be better than random
assert cv_mean > 0.5, "CV accuracy should be better than random!"
# Random guessing = 0.5 for binary classification

# Check 2: Results must be stable
assert stability['is_stable'], "Cross-validation results are unstable!"
# Unstable results suggest model is unreliable

print("\n✓ Cross-validation checks passed")  # All checks passed!


In [ ]:
# ============================================
# VALIDATION 3: Confusion Matrix Analysis
# ============================================

# Confusion matrix shows detailed breakdown of predictions
# For binary classification, it's a 2×2 matrix:
#                Predicted
#              Negative  Positive
# Actual Neg    TN       FP
# Actual Pos    FN       TP
#
# TN = True Negatives (correctly predicted negative)
# FP = False Positives (predicted positive but actually negative)
# FN = False Negatives (predicted negative but actually positive)
# TP = True Positives (correctly predicted positive)

# analyze_confusion_matrix() creates confusion matrix and calculates per-class metrics
cm_analysis = analyze_confusion_matrix(y_test.values, y_pred)
# Returns dictionary with confusion matrix and per-class metrics

print("Confusion Matrix Analysis:")
print(f"Confusion Matrix:\n{cm_analysis['confusion_matrix']}")  # Display the matrix

print(f"\nPer-Class Metrics:")
# Loop through each class and display its metrics
for class_name, class_metrics in cm_analysis['per_class_metrics'].items():
    # class_name = class label (e.g., '0' or '1', or 'benign'/'malignant')
    # class_metrics = dictionary with precision, recall, F1 for this class
    print(f"  {class_name}: Precision={class_metrics['precision']:.3f}, "
          f"Recall={class_metrics['recall']:.3f}, F1={class_metrics['f1_score']:.3f}")

# ============================================
# ASSERTIONS: Confusion Matrix Checks
# ============================================

# Check: Binary classification should have exactly 2 classes
assert len(cm_analysis['per_class_metrics']) == 2, "Should have 2 classes for binary classification"
# If not 2 classes, something is wrong with the data or model

print("\n✓ Confusion matrix analysis complete")  # Analysis finished!


## Performance Benchmarking

Let's benchmark the model's performance and compare with baseline.

**Why Benchmark?**
- **Training Time**: How long does it take to train? (important for large datasets)
- **Prediction Speed**: How fast can we make predictions? (important for real-time applications)
- **Baseline Comparison**: Is our model better than a simple baseline? (must outperform majority class predictor)

**Baseline Models for Classification:**
- **Majority Class Predictor**: Always predicts the most common class
- **Random Predictor**: Random predictions (worst case)
- **Stratified Random**: Predicts classes with same distribution as training data


In [ ]:
# ============================================
# PERFORMANCE BENCHMARKING: Measuring Speed
# ============================================

# benchmark_model_training() measures:
# - How long training takes
# - How long predictions take
# - How many predictions per second (throughput)
# - Model accuracy

# Parameters:
# - LogisticRegression(...): The model to benchmark (not yet trained)
# - X_train.values, y_train.values: Training data (NumPy arrays)
# - X_test.values, y_test.values: Test data (NumPy arrays)

benchmark_results = benchmark_model_training(
    LogisticRegression(max_iter=1000, random_state=42),  # Model class (will create new instance)
    X_train.values,  # Training features (convert DataFrame to NumPy array)
    y_train.values,  # Training targets (convert Series to NumPy array)
    X_test.values,  # Test features
    y_test.values  # Test targets
)
# Returns dictionary with timing and performance metrics

print("Performance Benchmark:")
# Display training time (how long it took to fit the model)
print(f"  Training Time: {benchmark_results['training_time']:.4f} seconds")

# Display prediction time (how long it took to predict on test set)
print(f"  Prediction Time: {benchmark_results['prediction_time']:.4f} seconds")

# Display throughput (how many predictions per second)
# Higher is better (faster predictions)
print(f"  Predictions per Second: {benchmark_results['predictions_per_second']:.0f}")

# Display model accuracy if available
if 'test_accuracy' in benchmark_results:
    print(f"  Test Accuracy: {benchmark_results['test_accuracy']:.3f}")


## Traceability

Let's extract feature importance and create traceability records.

**What is Traceability?**
Traceability means keeping records of:
- **What model was trained**: Algorithm, hyperparameters, dataset
- **What features were used**: Feature names, importance, coefficients
- **How well it performed**: Metrics, validation results
- **When it was trained**: Timestamp, version

**Why is it Important?**
- **Reproducibility**: Can recreate the same model later
- **Debugging**: Understand why model made certain predictions
- **Compliance**: Required for regulated industries (healthcare, finance)
- **Documentation**: Record of what was tried and what worked


In [ ]:
# ============================================
# FEATURE IMPORTANCE: Which Features Matter Most?
# ============================================

# In logistic regression, feature importance = absolute value of coefficients
# Larger coefficient (positive or negative) = more impact on log-odds
# extract_feature_importance_trace() extracts and organizes this information

feature_importance = extract_feature_importance_trace(
    model,  # The trained model (has .coef_ attribute)
    feature_names=X.columns.tolist()  # List of feature names
)
# Returns DataFrame with features sorted by importance (absolute coefficient value)

print("Feature Importance (by absolute coefficient):")
print(feature_importance.head(10))  # Display top 10 most important features

# ============================================
# VISUALIZING FEATURE IMPORTANCE
# ============================================

# Create horizontal bar chart showing top 10 feature importance
plt.figure(figsize=(10, 6))  # Figure size: 10×6 inches

# Get top 10 features
top_features = feature_importance.head(10)

# Create horizontal bar chart
# range(len(top_features)): Y-axis positions (0, 1, 2, ..., 9)
# top_features['importance']: Bar lengths (absolute coefficient values)
# align='center': Center bars on Y-axis positions
plt.barh(range(len(top_features)), top_features['importance'], align='center')

# Set Y-axis labels to feature names
plt.yticks(range(len(top_features)), top_features['feature'])

# Label axes
plt.xlabel('Absolute Coefficient Value')  # X-axis: importance magnitude
plt.title('Top 10 Feature Importance (Logistic Regression)')  # Chart title

# Invert Y-axis so most important feature is at top
plt.gca().invert_yaxis()  # gca() = get current axes

# Adjust layout to prevent label overlap
plt.tight_layout()
plt.show()  # Display the plot

# Interpretation:
# - Longer bars = more important features
# - Features at top have largest impact on predictions
# - Positive coefficients increase probability of positive class
# - Negative coefficients decrease probability of positive class


In [ ]:
# ============================================
# CREATING TRACEABILITY RECORD: Documenting Everything
# ============================================

# Create a dictionary containing all important information about the model
# This record can be saved and used later to reproduce or understand the model

trace_data = {
    # Model identification
    "model_type": "LogisticRegression",  # What algorithm was used
    "dataset": "Breast Cancer",  # What dataset was used
    
    # Dataset information
    "n_samples": len(X),  # Number of training samples (569)
    "n_features": X.shape[1],  # Number of features (30)
    
    # Model parameters (what the model learned)
    # dict(zip()) pairs feature names with their coefficients
    # Example: {'mean radius': 0.5, 'mean texture': -0.3, ...}
    "coefficients": dict(zip(X.columns, model.coef_[0])),
    "intercept": float(model.intercept_[0]),  # Log-odds intercept (β₀)
    
    # Feature importance (sorted by importance)
    # .to_dict('records') converts DataFrame to list of dictionaries
    "feature_importance": feature_importance.to_dict('records'),
    
    # Performance metrics (how well the model performed)
    "performance_metrics": {
        "accuracy": float(metrics['accuracy']),  # Overall correctness
        "precision": float(metrics['precision']),  # Precision of positive predictions
        "recall": float(metrics['recall']),  # Ability to find all positives
        "f1_score": float(metrics['f1_score']),  # Balanced metric
        "roc_auc": float(metrics.get('roc_auc', 0))  # Classification performance (0 if not calculated)
    },
    
    # Cross-validation results (robust performance estimate)
    "cross_validation": {
        "mean_accuracy": float(cv_mean),  # Average accuracy across folds
        "std_accuracy": float(cv_std)  # Standard deviation (variability)
    }
}

# ============================================
# SAVING TRACEABILITY DATA: Persisting Records
# ============================================

# save_traceability_data() saves the record to a JSON file
# File is timestamped to prevent overwriting previous records
# "logistic_regression_trace" is the base filename

trace_path = save_traceability_data(trace_data, "logistic_regression_trace")
# Returns path to saved file (e.g., "outputs/traceability/logistic_regression_trace_20251230_143022.json")

print(f"Traceability data saved to: {trace_path}")  # Show where file was saved

# This file can be loaded later to:
# - Reproduce the same model
# - Understand what features were important
# - Compare with other models
# - Document experiments for reports


## Visualization & Diagnostics

Let's visualize model performance with ROC curves, precision-recall curves, and confusion matrix.

**Why Visualize?**
- **See Performance**: Visual inspection reveals issues numbers miss
- **Understand Trade-offs**: ROC and PR curves show precision-recall trade-offs
- **Diagnose Problems**: Confusion matrix shows where model makes mistakes
- **Communicate Results**: Visuals are easier to understand than numbers

**Key Visualizations:**
1. **Confusion Matrix**: Shows correct vs incorrect predictions per class
2. **ROC Curve**: Shows true positive rate vs false positive rate
3. **Precision-Recall Curve**: Shows precision vs recall at different thresholds


In [ ]:
# ============================================
# VISUALIZATION 1: Confusion Matrix
# ============================================

# plot_confusion_matrix() creates a visual confusion matrix
# Shows how many predictions were correct/incorrect for each class

plot_confusion_matrix(
    y_test.values,  # True class labels
    y_pred,  # Predicted class labels
    class_names=cancer.target_names.tolist(),  # Class names: ['malignant', 'benign']
    title="Logistic Regression Confusion Matrix"  # Plot title
)
# Creates a heatmap showing:
# - True Negatives (TN): Correctly predicted benign
# - False Positives (FP): Predicted malignant but actually benign
# - False Negatives (FN): Predicted benign but actually malignant
# - True Positives (TP): Correctly predicted malignant

# Interpretation:
# - Diagonal cells (TN, TP) = correct predictions (should be high)
# - Off-diagonal cells (FP, FN) = errors (should be low)
# - FP = Type I error (false alarm)
# - FN = Type II error (missed detection)


In [ ]:
# ============================================
# VISUALIZATION 2: ROC Curve
# ============================================

# ROC (Receiver Operating Characteristic) curve shows classification performance
# X-axis: False Positive Rate (FPR) = FP / (FP + TN)
# Y-axis: True Positive Rate (TPR) = TP / (TP + FN) = Recall
# Shows trade-off between sensitivity and specificity

# plot_roc_curve() creates the ROC curve and calculates AUC
roc_auc, fpr, tpr = plot_roc_curve(
    y_test.values,  # True class labels
    y_pred_proba,  # Predicted probabilities (not class predictions!)
    title="Logistic Regression ROC Curve"  # Plot title
)
# Returns:
# - roc_auc: Area Under the Curve (0 to 1, higher is better)
# - fpr: False Positive Rate values for the curve
# - tpr: True Positive Rate values for the curve

print(f"ROC-AUC Score: {roc_auc:.3f}")  # Display AUC score

# Interpretation:
# - ROC-AUC = 1.0: Perfect classifier (all positives ranked above negatives)
# - ROC-AUC = 0.5: Random classifier (no better than guessing)
# - ROC-AUC < 0.5: Worse than random (model is inverted)
# - Higher AUC = better ability to distinguish between classes
# - Curve closer to top-left corner = better performance


In [ ]:
# ============================================
# VISUALIZATION 3: Precision-Recall Curve
# ============================================

# Precision-Recall curve shows precision vs recall at different thresholds
# X-axis: Recall (True Positive Rate) = TP / (TP + FN)
# Y-axis: Precision = TP / (TP + FP)
# Useful when classes are imbalanced (more informative than ROC for imbalanced data)

# plot_precision_recall_curve() creates the PR curve and calculates average precision
avg_precision, precision, recall = plot_precision_recall_curve(
    y_test.values,  # True class labels
    y_pred_proba,  # Predicted probabilities
    title="Logistic Regression Precision-Recall Curve"  # Plot title
)
# Returns:
# - avg_precision: Average precision score (0 to 1, higher is better)
# - precision: Precision values for the curve
# - recall: Recall values for the curve

print(f"Average Precision: {avg_precision:.3f}")  # Display average precision

# Interpretation:
# - Average Precision = 1.0: Perfect classifier
# - Average Precision = 0.5: Moderate performance
# - Average Precision < 0.5: Poor performance
# - Curve closer to top-right corner = better performance
# - Better than ROC for imbalanced datasets (when one class is rare)


## Real-World Application

Let's apply logistic regression to multiclass classification and hyperparameter tuning.

**Multiclass Classification:**
- Logistic regression can handle more than 2 classes
- Uses one-vs-rest (OvR) or multinomial (softmax) approach
- Each class gets its own set of coefficients

**Hyperparameter Tuning:**
- **C**: Regularization strength (inverse of regularization parameter)
  - Smaller C = stronger regularization (prevents overfitting)
  - Larger C = weaker regularization (fits training data more closely)
- **penalty**: Type of regularization ('l1' or 'l2')
  - L1 (Lasso): Can set coefficients to zero (feature selection)
  - L2 (Ridge): Shrinks coefficients but doesn't eliminate features


In [ ]:
# ============================================
# MULTICLASS CLASSIFICATION: Iris Dataset
# ============================================

# load_iris() loads the Iris flower dataset (3 classes: setosa, versicolor, virginica)
iris = load_iris()
X_iris = pd.DataFrame(iris.data, columns=iris.feature_names)  # Features: 4 measurements
y_iris = pd.Series(iris.target, name='Species')  # Target: 3 flower species

# Scale features and split data
X_iris_scaled, _ = scale_features(X_iris, fit=True)  # Normalize features
X_iris_train, X_iris_test, y_iris_train, y_iris_test = split_data(
    X_iris_scaled, y_iris, test_size=0.2, random_state=42
)

# ============================================
# TRAIN MULTICLASS LOGISTIC REGRESSION
# ============================================

# multi_class='multinomial' uses softmax regression (multinomial logistic regression)
# This treats all classes together (better than one-vs-rest for multiclass)
# Alternative: multi_class='ovr' (one-vs-rest) trains separate binary classifiers
model_multi = LogisticRegression(max_iter=1000, random_state=42, multi_class='multinomial')
model_multi.fit(X_iris_train, y_iris_train)  # Train on Iris data

# Make predictions and evaluate
y_iris_pred = model_multi.predict(X_iris_test)  # Predict class labels
iris_accuracy = accuracy_score(y_iris_test, y_iris_pred)  # Calculate accuracy

print("Multiclass Classification (Iris Dataset):")
print(f"  Accuracy: {iris_accuracy:.3f}")  # Overall correctness
print(f"  Classes: {iris.target_names.tolist()}")  # Class names: ['setosa', 'versicolor', 'virginica']

# ============================================
# HYPERPARAMETER TUNING: Finding Best Parameters
# ============================================

# GridSearchCV tries all combinations of hyperparameters and finds the best
# param_grid defines the hyperparameters to search:
# - C: Regularization strength [0.001, 0.01, 0.1, 1, 10, 100]
#   Smaller C = stronger regularization, larger C = weaker regularization
# - penalty: Type of regularization ['l1', 'l2']
#   L1 can eliminate features, L2 shrinks coefficients

param_grid = {'C': [0.001, 0.01, 0.1, 1, 10, 100], 'penalty': ['l1', 'l2']}

# GridSearchCV performs cross-validation for each hyperparameter combination
# cv=5: 5-fold cross-validation
# scoring='accuracy': Use accuracy to evaluate each combination
# solver='liblinear': Optimization algorithm (supports both L1 and L2)
grid_search = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=42, solver='liblinear'),
    param_grid,  # Hyperparameters to search
    cv=5,  # 5-fold cross-validation
    scoring='accuracy'  # Evaluation metric
)
grid_search.fit(X_train, y_train)  # Train and evaluate all combinations

# Display best hyperparameters found
print(f"\nBest hyperparameters: {grid_search.best_params_}")  # Best C and penalty
print(f"Best CV accuracy: {grid_search.best_score_:.3f}")  # Best cross-validation accuracy

# grid_search.best_estimator_ is the model trained with best hyperparameters
# Can use this for final predictions: grid_search.best_estimator_.predict(X_test)


## Summary & Key Takeaways

### Key Concepts Learned

1. **Logistic Regression Basics**
   - Models probability of class membership using sigmoid function
   - Uses maximum likelihood estimation
   - Provides interpretable coefficients (log-odds)

2. **Model Evaluation**
   - Accuracy, precision, recall, F1-score for classification
   - ROC-AUC for binary classification performance
   - Confusion matrix for detailed error analysis

3. **Best Practices**
   - Scale features for convergence
   - Use cross-validation for robust evaluation
   - Tune regularization parameter (C)
   - Check for class imbalance

### When to Use Logistic Regression

✅ **Good for:**
- Binary and multiclass classification
- When probability estimates are needed
- Interpretability is important
- Baseline classifier for comparison
- Linear decision boundaries are sufficient

❌ **Not ideal for:**
- Non-linear decision boundaries (use kernel methods or neural networks)
- Very high-dimensional sparse data (use Naive Bayes)
- Complex feature interactions (use tree-based methods)

### Next Steps

- Try **Regularized Logistic Regression** (L1/L2) for feature selection
- Explore **Polynomial Features** for non-linear relationships
- Compare with **Support Vector Machines** for similar use cases
- Consider **Neural Networks** for complex non-linear patterns
